In [1]:
import pandas as pd, numpy as np, re
from pathlib import Path

BASE = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data")
PAST = BASE / "Combined Past Holdings"
OUT = BASE / "all_holdings_2017_2025.csv"

def to_float(s):
    if s is None:
        return np.nan
    t = str(s).strip()
    if t in {"", "nan", "NA", "N/A", "-", "--", "—"}:
        return np.nan
    t = t.replace("(", "-").replace(")", "")
    t = re.sub(r"[^0-9.\-]", "", t)
    if t.count(".") > 1:
        first, *rest = t.split(".")
        t = first + "." + "".join(rest)
    try:
        return float(t)
    except:
        return np.nan

def load_soi(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    req = {"etf_ticker","etf_name","company_ticker","name_normalized","value"}
    if not req.issubset(df.columns):
        raise ValueError(f"Missing columns in {path.name}: {req - set(df.columns)}")
    df["value"] = df["value"].map(to_float).fillna(0.0)
    if "total_overall_value" in df.columns:
        df["total_overall_value"] = df["total_overall_value"].map(to_float)
    y = re.search(r"soi_(\d{4})_final\.csv$", path.name, flags=re.I)
    if not y:
        raise ValueError(f"Year parse failed for {path.name}")
    year = int(y.group(1))
    gsum = df.groupby("etf_ticker")["value"].transform("sum")
    if "total_overall_value" in df.columns:
        tv = df.groupby("etf_ticker")["total_overall_value"].transform(lambda s: s.dropna().max() if s.notna().any() else np.nan)
        total = np.where(pd.notna(tv) & (tv > 0), tv, gsum)
    else:
        total = gsum
    df = df[["etf_ticker","name_normalized","company_ticker","value"]].copy()
    df["total_value"] = total
    df = df[df["total_value"] > 0]
    df["weight(%)"] = 100.0 * df["value"] / df["total_value"]
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = f"{year}-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

def load_2025(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = None
    for cand in ["weight (%)","weight(%)","weight_percent","weight"]:
        if cand in df.columns:
            wcol = cand
            break
    if wcol is None:
        raise ValueError("No weight column found for 2025")
    df["weight(%)"] = df[wcol].apply(lambda x: to_float(str(x).replace("%","")))
    df = df[["etf_ticker","name_normalized","company_ticker","weight(%)"]].copy()
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = "2025-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

past_files = sorted([p for p in PAST.glob("soi_20*_final.csv") if p.is_file()])
past_frames = [load_soi(p) for p in past_files]
cur_2025 = load_2025(BASE / "holdings_2025_final.csv")

all_holdings = pd.concat(past_frames + [cur_2025], ignore_index=True)
all_holdings["weight(%)"] = all_holdings["weight(%)"].astype(float).round(6)
all_holdings = all_holdings.sort_values(["ETF_TICKER","date","weight(%)"], ascending=[True,True,False]).reset_index(drop=True)
all_holdings.to_csv(OUT, index=False)
print(f"Wrote {OUT}")


Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/all_holdings_2017_2025.csv
